In [5]:
from dotenv import load_dotenv
import os
load_dotenv()
print("dotenv loaded successfully")



dotenv loaded successfully


In [11]:
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
# llm = init_chat_model("deepseek-r1-distill-llama-70b")
llm = ChatGroq(model ="llama-3.3-70b-versatile",api_key=os.getenv("GROQ_API_KEY"))

## Different ways to create tools

In [12]:
response=llm.invoke("Hi")
print(response.content)

It's nice to meet you. Is there something I can help you with or would you like to chat?


### Tool using decorators

In [14]:
from langchain.tools import tool

@tool
def multiply(a:int,b:int)->int:
    """Multiply two integers""" #-> Must pass this 
    return a*b


### Tool using structred 

In [ ]:
# Import required libraries
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from typing import Literal
import datetime
import requests  #if you want real API integration

# -------------------------------
# Define the input schema for the tool
# -------------------------------
class WeatherInput(BaseModel):
    """
    This defines the expected input structure for the weather tool.
    """
    city: str = Field(description="City for which to fetch weather")
    date: str = Field(description="Date for the weather forecast in YYYY-MM-DD format"
    )

# -------------------------------
# Define the output schema for clarity
# -------------------------------
class WeatherOutput(BaseModel):
    """
    This defines the expected output structure of the weather tool.
    """
    city: str
    date: str
    forecast: str

# -------------------------------
# Define the actual function that fetches weather
# -------------------------------
def get_weather(city: str, date: str) -> WeatherOutput:
    """
    Returns a weather forecast for a given city and date.

    In a real implementation, you could call a weather API here
    (like OpenWeatherMap or WeatherAPI). Currently, this returns a dummy response.
    """
    # Validate the date format
    try:
        datetime.datetime.strptime(date, "%Y-%m-%d")
    except ValueError:
        raise ValueError("Date must be in YYYY-MM-DD format")
    
    # Dummy forecast logic
    forecast = "Sunny with a chance of clouds"  # Replace with real API call if needed

    # Return structured output
    return WeatherOutput(city=city, date=date, forecast=forecast)

# -------------------------------
# Wrap the function in a LangChain StructuredTool
# -------------------------------
weather_tool = StructuredTool.from_function(
    func=get_weather,
    name="get_weather",  # Tool name used by agents
    description="Fetches real-time or forecasted weather data for a city on a given date",
    args_schema=WeatherInput,  # Input validation using Pydantic
    
)

# -------------------------------
# Example usage
# -------------------------------

# Example input
data_dict = {"city": "London", "date": "2025-10-22"}
input_data = WeatherInput(**data_dict)

# Call the tool
result = weather_tool.invoke(input_data.model_dump())
  # StructuredTool expects a dict

# Print result
print(result.model_dump_json())
result

{"city":"London","date":"2025-10-22","forecast":"Sunny with a chance of clouds"}


WeatherOutput(city='London', date='2025-10-22', forecast='Sunny with a chance of clouds')

{"city":"London","date":"2025-10-22","forecast":"Sunny with a chance of clouds"}
